<a href="https://colab.research.google.com/github/Samuelisusername/Large-Language-Models/blob/main/Tutorial_2_NPLM_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import trange
import numpy as np
import nltk
from nltk.corpus import brown
from nltk import WhitespaceTokenizer
import pandas as pd
from torch.utils.data import Dataset, DataLoader

In [ ]:
N_GRAM = 3
CONTEXT_SIZE = N_GRAM-1

## Dataset

In [ ]:
# download dataset
# add EOS and lower corpus

nltk.download('brown')
sentences = brown.sents()
flattened_sentences = [' '.join(map(str.lower, sentence))+ " <EOS>" for sentence in sentences]
corpus = flattened_sentences[:1000]

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


In [ ]:
len(flattened_sentences)

57340

In [ ]:
flattened_sentences[0]

"the fulton county grand jury said friday an investigation of atlanta's recent primary election produced `` no evidence '' that any irregularities took place . <EOS>"

In [ ]:
# tokenize by whitespace

tokens=[]
tokenizer = WhitespaceTokenizer()
for sentence in corpus:
  tokens.extend(tokenizer.tokenize(sentence))
tokens = list(set(tokens))
word_to_idx = {word: i for i, word in enumerate(tokens)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

In [ ]:
# create n-grams dataset

contexts = []
targets = []

for sentence in corpus:
  inputs = [word for word in sentence.split()]
  if len(inputs) < N_GRAM:
    continue
  for i in range(len(inputs) - N_GRAM + 1):
    context = inputs[i:i + N_GRAM - 1]
    target = inputs[i + N_GRAM - 1]
    contexts.append(context)
    targets.append(target)


In [ ]:
text_labels_df = pd.DataFrame({'context': contexts, 'target': targets})
text_labels_df

,context,target
0,"[the, fulton]",county
1,"[fulton, county]",grand
2,"[county, grand]",jury
3,"[grand, jury]",said
4,"[jury, said]",friday
...,...,...
21074,"[where, we]",have
21075,"[we, have]",200
21076,"[have, 200]",;
21077,"[200, ;]",;


## NPLM Model

In [ ]:
class NPLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim):
        super(NPLM, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.activation = nn.Tanh()
        self.linear2 = nn.Linear(hidden_dim, vocab_size, bias=False)
        self.linear3 = nn.Linear(context_size * embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs)
        embeds = embeds.view(inputs.size(0), -1)
        out1 = self.activation(self.linear1(embeds))
        out1 = self.linear2(out1)
        out2 = self.linear3(embeds)

        return out1+out2

In [ ]:
# Hyperparameters
VOCAB_SIZE = len(tokens)
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
LEARNING_RATE = 0.01
EPOCHS = 50
BATCH_SIZE = 32

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define the model
model = NPLM(VOCAB_SIZE, EMBEDDING_DIM, CONTEXT_SIZE, HIDDEN_DIM).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Parameters

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)
num_params = count_parameters(model)
print(f'Number of parameters in the model: {num_params}')

Number of parameters in the model: 1106417


## Parameters Manually

In [ ]:
def count_parameters_manually(vocab_size, embd_dim, hidden_dim, context_size):
  embedding_params = vocab_size * embd_dim
  linear1_params = context_size * embd_dim * hidden_dim + hidden_dim
  linear2_params = hidden_dim * vocab_size
  linear3_params = context_size * embd_dim * vocab_size + vocab_size
  num_params_manually = embedding_params + linear1_params + linear2_params + linear3_params
  return num_params_manually

num_params_manually = count_parameters_manually(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, CONTEXT_SIZE)
print(f'Number of parameters in the model: {num_params_manually}')

Number of parameters in the model: 1106417


## Training

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context = self.data.iloc[idx]['context']
        target = self.data.iloc[idx]['target']

        context_ids = [word_to_idx[word] for word in context]
        target_ids = word_to_idx[target]

        context_tensor = torch.tensor(context_ids).to(device)
        target_tensor = torch.tensor(target_ids).to(device)

        return context_tensor, target_tensor

In [ ]:
dataset = CustomDataset(text_labels_df)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Training
for epoch in trange(EPOCHS):
    total_loss = 0
    for inputs, labels in dataloader:
      optimizer.zero_grad()
      output = model(inputs.to(device))
      loss = criterion(output, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    if (epoch + 1) % 5 == 0:
      print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}')

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5, Loss: 3.2159
Epoch 10, Loss: 2.6698
Epoch 15, Loss: 2.4537
Epoch 20, Loss: 2.3497
Epoch 25, Loss: 2.3227
Epoch 30, Loss: 2.2632
Epoch 35, Loss: 2.2481
Epoch 40, Loss: 2.2167
Epoch 45, Loss: 2.2037
Epoch 50, Loss: 2.2406


## Testing

In [ ]:
# Testing 3-gram
model.eval()

def predict_word(context_var,k):
  with torch.no_grad():
    output = model(context_var.unsqueeze(0))
    probabilities = torch.softmax(output, dim=1)
    if k==0:
      predicted_idx = torch.argmax(probabilities,dim=1).item()
    else:
      top_k_probs, top_k_indices = torch.topk(probabilities.squeeze(0), k)
      top_k_probs = top_k_probs / torch.sum(top_k_probs)
      sampled_idx = torch.multinomial(top_k_probs, 1).item()
      predicted_idx = top_k_indices[sampled_idx].item()
    predicted_word = idx_to_word[predicted_idx]
    return predicted_word

def predict_sentence(sentence, length, k=0):
  s = ' '.join(sentence)
  context = sentence[-CONTEXT_SIZE:]
  for i in range(length-len(sentence)):
    context_var = torch.tensor([word_to_idx[word] for word in context]).to(device)
    w = predict_word(context_var,k)
    if w=='<EOS>':
      break
    context = context[1:]
    context.append(w)
    s = s + ' '+ w
  print(f'Sentence predicted: {s}')

predict_sentence(['they', 'have'], 8, 40)

Sentence predicted: they have essential service but is expected to


In [ ]:
predict_sentence(['she', 'is'], 7, 100)

Sentence predicted: she is not expected for some leadership
